In [ ]:
!pip install ultralytics opencv-python numpy

In [ ]:
import cv2
import numpy as np
import math
from ultralytics import YOLO

In [ ]:
model = YOLO("yolov8n.pt")
print("Model loaded")

Model loaded


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving rescue_video.mp4 to rescue_video.mp4


In [ ]:
import cv2
from google.colab.patches import cv2_imshow
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

video_path = "rescue_video.mp4"
cap = cv2.VideoCapture(video_path)

frame_count = 0
saved_images = 0

while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    frame_count += 1

    # Skip some frames for faster processing
    if frame_count % 5 != 0:
        continue

    frame = cv2.resize(frame, (640,640))

    results = model(frame, conf=0.4)

    # Skip frames with no detections
    if len(results[0].boxes) == 0:
        continue

    annotated_frame = results[0].plot()

    cv2_imshow(annotated_frame)

    # Save detection frame
    filename = f"detection_{saved_images}.png"
    cv2.imwrite(filename, annotated_frame)

    print("Saved:", filename)

    saved_images += 1

    # Stop after saving 5 images
    if saved_images == 5:
        break

cap.release()

In [ ]:
!pip install ultralytics opencv-python numpy

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from google.colab.patches import cv2_imshow

In [ ]:
model = YOLO("yolov8n.pt")
print("Model loaded successfully")

Model loaded successfully


In [ ]:
from google.colab import files
files.upload()

Saving rescue_video.mp4 to rescue_video (1).mp4


In [ ]:
def process_drone_feed(source):

    cap = cv2.VideoCapture(source)

    frame_count = 0

    while cap.isOpened():

        ret, frame = cap.read()

        if not ret:
            break

        frame_count += 1

        # show only some frames (important for Colab)
        if frame_count % 10 != 0:
            continue

        frame = cv2.resize(frame, (640,640))

        # YOLO inference
        results = model(frame, conf=0.3)

        for r in results:

            boxes = r.boxes.xyxy.cpu().numpy()
            scores = r.boxes.conf.cpu().numpy()
            classes = r.boxes.cls.cpu().numpy()

            for box, score, cls in zip(boxes, scores, classes):

                x1, y1, x2, y2 = map(int, box)

                label = model.names[int(cls)]

                print("Object:", label)
                print("Bounding Box:", x1, y1, x2, y2)
                print("Confidence:", float(score))

                cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)

                cv2.putText(frame,
                            f"{label} {score:.2f}",
                            (x1,y1-10),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.6,
                            (0,255,0),
                            2)

        cv2_imshow(frame)

    cap.release()